In [3]:
# 1. Import Interactive Visualization Dependencies

from __future__ import annotations



import importlib

import json

import math

import subprocess

import sys

from dataclasses import dataclass, asdict

from datetime import datetime, timezone

from functools import lru_cache

from pathlib import Path



missing_packages = [name for name in ('ipywidgets', 'mpmath') if importlib.util.find_spec(name) is None]

if missing_packages:

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])



import ipywidgets as widgets

import matplotlib.pyplot as plt

import numpy as np

from IPython.display import Markdown, display

from matplotlib.colors import Normalize, TwoSlopeNorm

from matplotlib.transforms import Affine2D

from mpmath import mp



REPO_ROOT = Path('/workspaces/newQFE')

MODULAR_DIR = REPO_ROOT / 'modular analogue'

RMT_SCRIPTS = REPO_ROOT / 'responsible_method_tests' / 'scripts'

K_FROM_CONTINUUM = REPO_ROOT / 'K_from_continuum'



for path in (RMT_SCRIPTS, K_FROM_CONTINUUM):

    if str(path) not in sys.path:

        sys.path.insert(0, str(path))



from generate_pointwise_manifold_dataset import _ising_torus_shape, _modular_tau

from workflow_common import exact_triangular_ising_beta



plt.ioff()

NOTEBOOK_OUTPUT_DIR = MODULAR_DIR / 'interactive_outputs'

NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JOURNAL_ARTIFACT = MODULAR_DIR / 'interactive_notebook_journal.md'

np.set_printoptions(precision=6, suppress=True)

print('interactive output dir:', NOTEBOOK_OUTPUT_DIR)


  Using cached mpmath-1.4.1-py3-none-any.whl.metadata (9.1 kB)
Using cached mpmath-1.4.1-py3-none-any.whl (567 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


interactive output dir: /workspaces/newQFE/modular analogue/interactive_outputs


In [4]:
# 2. Define the 1-1-1 and 4-5-6 Geometry Inputs

@dataclass(frozen=True)

class CaseSpec:

    key: str

    label: str

    truth_r1: float

    truth_r2: float

    lattice_geometry: tuple[int, int, int, int]

    tau_real_halfspan: float

    tau_imag_halfspan: float

    multiplier_min: float = 0.6

    multiplier_max: float = 1.4





BASE_CASES: dict[str, CaseSpec] = {

    'iso111': CaseSpec(

        key='iso111',

        label='1-1-1 control',

        truth_r1=1.0,

        truth_r2=1.0,

        lattice_geometry=(12, 12, 0, 0),

        tau_real_halfspan=0.08,

        tau_imag_halfspan=0.20,

    ),

    'acute456': CaseSpec(

        key='acute456',

        label='4-5-6 small target',

        truth_r1=4.702782819756,

        truth_r2=7.353910143333,

        lattice_geometry=(66, 66, 33, 11),

        tau_real_halfspan=0.08,

        tau_imag_halfspan=0.20,

    ),

}





def target_tau_from_geometry(geometry: tuple[int, int, int, int]) -> complex:

    tau, _ = _modular_tau(*geometry)

    return complex(float(tau.real), float(tau.imag))





def triangle_overlay_vertices(tau: complex) -> np.ndarray:

    return np.array(

        [

            [0.0, 0.0],

            [1.0, 0.0],

            [float(tau.real), float(tau.imag)],

        ],

        dtype=float,

    )





display(

    Markdown(

        '\n'.join(

            [

                '## Baseline cases',

                '',

                f"- iso111 target geometry: {BASE_CASES['iso111'].lattice_geometry}, truth = (1.0, 1.0)",

                f"- acute456 target geometry: {BASE_CASES['acute456'].lattice_geometry}, truth = ({BASE_CASES['acute456'].truth_r1:.6f}, {BASE_CASES['acute456'].truth_r2:.6f})",

            ]

        )

    )

)


## Baseline cases

- iso111 target geometry: (12, 12, 0, 0), truth = (1.0, 1.0)
- acute456 target geometry: (66, 66, 33, 11), truth = (4.702783, 7.353910)

In [5]:
# 3. Build Parametric Geometry Transform Functions

def build_affine(

    rotation_deg: float = 0.0,

    translate_x: float = 0.0,

    translate_y: float = 0.0,

    scale_factor: float = 1.0,

    reflect_x: bool = False,

    reflect_y: bool = False,

) -> Affine2D:

    sx = -float(scale_factor) if reflect_x else float(scale_factor)

    sy = -float(scale_factor) if reflect_y else float(scale_factor)

    affine = Affine2D()

    affine.scale(sx, sy)

    affine.rotate_deg(float(rotation_deg))

    affine.translate(float(translate_x), float(translate_y))

    return affine





def apply_affine(points: np.ndarray, **kwargs: float | bool) -> np.ndarray:

    affine = build_affine(**kwargs)

    return affine.transform(np.asarray(points, dtype=float))





def tau_eff_from_couplings(r1: float, r2: float) -> dict[str, float | complex]:

    couplings = {'k1': float(r1), 'k2': float(r2), 'k3': 1.0}

    beta_c = float(exact_triangular_ising_beta(couplings))

    alpha = math.atan2(1.0, math.sinh(2.0 * beta_c * couplings['k2']))

    beta_angle = math.atan2(1.0, math.sinh(2.0 * beta_c * couplings['k1']))

    gamma = math.atan2(1.0, math.sinh(2.0 * beta_c * couplings['k3']))

    side_ratio = math.sin(alpha) / math.sin(beta_angle)

    tau_eff = complex(side_ratio * math.cos(gamma), side_ratio * math.sin(gamma))

    return {

        'tau_eff': tau_eff,

        'beta_c': beta_c,

        'alpha': alpha,

        'beta_angle': beta_angle,

        'gamma': gamma,

    }





def make_case_descriptor(case_key: str, label: str, r1: float, r2: float) -> dict[str, object]:

    base = BASE_CASES[case_key]

    tau_data = tau_eff_from_couplings(r1, r2)

    return {

        'key': case_key,

        'label': label,

        'r1': float(r1),

        'r2': float(r2),

        'tau': tau_data['tau_eff'],

        'beta_c': tau_data['beta_c'],

        'angles_deg': {

            'alpha': math.degrees(float(tau_data['alpha'])),

            'beta': math.degrees(float(tau_data['beta_angle'])),

            'gamma': math.degrees(float(tau_data['gamma'])),

        },

        'reference_geometry': base.lattice_geometry,

        'reference_tau': target_tau_from_geometry(base.lattice_geometry),

        'tau_real_halfspan': base.tau_real_halfspan,

        'tau_imag_halfspan': base.tau_imag_halfspan,

        'multiplier_min': base.multiplier_min,

        'multiplier_max': base.multiplier_max,

    }


In [6]:
# 4. Generate the Heatmap Grid and Field Computation

OBSERVABLE_KEYS = ('v4', 'u4', 'w4', 'v4_over_u4', 'w4_over_u4')

SIGNED_METRICS = tuple(f'{key}_log_residual' for key in OBSERVABLE_KEYS)

POSITIVE_METRICS = tuple(f'{key}_chi2' for key in OBSERVABLE_KEYS) + ('corr_score', 'ratio_score', 'total_score')

METRIC_LABELS = {

    'v4_log_residual': 'log residual[v/4 anchor-ratio]',

    'u4_log_residual': 'log residual[u/4 anchor-ratio]',

    'w4_log_residual': 'log residual[w/4 anchor-ratio]',

    'v4_over_u4_log_residual': 'log residual[(v/4)/(u/4)]',

    'w4_over_u4_log_residual': 'log residual[(w/4)/(u/4)]',

    'v4_chi2': 'chi2[v/4 anchor-ratio]',

    'u4_chi2': 'chi2[u/4 anchor-ratio]',

    'w4_chi2': 'chi2[w/4 anchor-ratio]',

    'v4_over_u4_chi2': 'chi2[(v/4)/(u/4)]',

    'w4_over_u4_chi2': 'chi2[(w/4)/(u/4)]',

    'corr_score': 'corr score',

    'ratio_score': 'ratio score',

    'total_score': 'total score',

}





def quarter_points(tau: complex) -> dict[str, complex]:

    return {

        'v4': complex(0.25, 0.0),

        'u4': 0.25 * tau,

        'w4': 0.25 * (1.0 + tau),

    }





@lru_cache(maxsize=4096)

def q_and_theta1p0(tau_real: float, tau_imag: float, mp_dps: int) -> tuple[mp.mpc, mp.mpf]:

    mp.dps = max(int(mp_dps), 30)

    tau_mp = mp.mpc(float(tau_real), float(tau_imag))

    q = mp.e ** (mp.pi * 1j * tau_mp)

    theta1p0 = mp.diff(lambda zz: mp.jtheta(1, zz, q), mp.mpf('0.0'))

    return q, theta1p0





@lru_cache(maxsize=4096)

def anchored_observable_tuple(tau_real: float, tau_imag: float, anchor_denominator: float, mp_dps: int) -> tuple[float, ...]:

    tau = complex(float(tau_real), float(tau_imag))

    q, theta1p0 = q_and_theta1p0(float(tau.real), float(tau.imag), int(mp_dps))

    anchor = tau / float(anchor_denominator)

    anchor_value = float(_ising_torus_shape(anchor, tau, theta1p0, q))

    if not np.isfinite(anchor_value) or anchor_value <= 0.0:

        raise ValueError(f'invalid anchor value at tau={tau}')

    points = quarter_points(tau)

    values: dict[str, float] = {}

    for key, nu in points.items():

        point_value = float(_ising_torus_shape(nu, tau, theta1p0, q))

        if not np.isfinite(point_value) or point_value <= 0.0:

            raise ValueError(f'invalid point value for {key} at tau={tau}')

        values[key] = float(point_value / anchor_value)

    values['v4_over_u4'] = float(values['v4'] / values['u4'])

    values['w4_over_u4'] = float(values['w4'] / values['u4'])

    return tuple(float(values[key]) for key in OBSERVABLE_KEYS)





def anchored_observables(tau: complex, anchor_denominator: float = 8.0, mp_dps: int = 40) -> dict[str, float]:

    rounded = (round(float(tau.real), 12), round(float(tau.imag), 12), float(anchor_denominator), int(mp_dps))

    values = anchored_observable_tuple(*rounded)

    return {key: float(value) for key, value in zip(OBSERVABLE_KEYS, values)}





def score_candidate_tau(candidate_tau: complex, target_values: dict[str, float], anchor_denominator: float = 8.0, mp_dps: int = 40) -> dict[str, float]:

    candidate_values = anchored_observables(candidate_tau, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

    row: dict[str, float] = {}

    corr_score = 0.0

    ratio_score = 0.0

    for key in OBSERVABLE_KEYS:

        residual = math.log(candidate_values[key]) - math.log(target_values[key])

        row[f'{key}_log_residual'] = residual

        row[f'{key}_chi2'] = residual * residual

        if key in {'v4', 'u4', 'w4'}:

            corr_score += residual * residual

        else:

            ratio_score += residual * residual

    row['corr_score'] = corr_score

    row['ratio_score'] = ratio_score

    row['total_score'] = corr_score + ratio_score

    return row





def build_tau_grid(case: dict[str, object], grid_size: int = 25, anchor_denominator: float = 8.0, mp_dps: int = 40) -> dict[str, object]:

    target_tau = complex(case['tau'])

    target_values = anchored_observables(target_tau, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

    xs = np.linspace(target_tau.real - float(case['tau_real_halfspan']), target_tau.real + float(case['tau_real_halfspan']), int(grid_size))

    ys = np.linspace(max(1.0e-3, target_tau.imag - float(case['tau_imag_halfspan'])), target_tau.imag + float(case['tau_imag_halfspan']), int(grid_size))

    fields = {metric: np.full((len(xs), len(ys)), np.nan, dtype=float) for metric in SIGNED_METRICS + POSITIVE_METRICS}

    for i, x_value in enumerate(xs):

        for j, y_value in enumerate(ys):

            row = score_candidate_tau(complex(float(x_value), float(y_value)), target_values, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

            for metric, metric_value in row.items():

                fields[metric][i, j] = float(metric_value)

    return {

        'space': 'tau',

        'x_values': xs,

        'y_values': ys,

        'target_xy': (float(target_tau.real), float(target_tau.imag)),

        'fields': fields,

        'overlay_vertices': triangle_overlay_vertices(target_tau),

        'descriptor': case,

    }





def build_rgrid(case: dict[str, object], grid_size: int = 25, anchor_denominator: float = 8.0, mp_dps: int = 40) -> dict[str, object]:

    target_tau = complex(case['tau'])

    target_values = anchored_observables(target_tau, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

    r1_target = float(case['r1'])

    r2_target = float(case['r2'])

    xs = np.linspace(float(case['multiplier_min']) * r1_target, float(case['multiplier_max']) * r1_target, int(grid_size))

    ys = np.linspace(float(case['multiplier_min']) * r2_target, float(case['multiplier_max']) * r2_target, int(grid_size))

    fields = {metric: np.full((len(xs), len(ys)), np.nan, dtype=float) for metric in SIGNED_METRICS + POSITIVE_METRICS}

    for i, r1_value in enumerate(xs):

        for j, r2_value in enumerate(ys):

            candidate_tau = complex(tau_eff_from_couplings(float(r1_value), float(r2_value))['tau_eff'])

            row = score_candidate_tau(candidate_tau, target_values, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

            for metric, metric_value in row.items():

                fields[metric][i, j] = float(metric_value)

    return {

        'space': 'rgrid',

        'x_values': xs,

        'y_values': ys,

        'target_xy': (r1_target, r2_target),

        'fields': fields,

        'overlay_vertices': triangle_overlay_vertices(target_tau),

        'descriptor': case,

    }


In [7]:
# 5. Render Synchronized Heatmaps for Both Cases

def grid_extent(values: np.ndarray) -> tuple[float, float]:

    if len(values) <= 1:

        return float(values[0]) - 0.5, float(values[0]) + 0.5

    step = float(np.median(np.diff(values)))

    return float(values[0] - 0.5 * step), float(values[-1] + 0.5 * step)





def normalize_field(field: np.ndarray, signed: bool) -> np.ndarray:

    finite = field[np.isfinite(field)]

    if finite.size == 0:

        return np.zeros_like(field)

    scale = float(np.nanmax(np.abs(finite))) if signed else float(np.nanmax(finite))

    if scale <= 0.0:

        scale = 1.0

    return np.asarray(field, dtype=float) / scale





def trough_orientation(field: np.ndarray, x_values: np.ndarray, y_values: np.ndarray, signed: bool) -> dict[str, float | tuple[float, float]]:

    score = np.abs(field) if signed else np.asarray(field, dtype=float)

    best_index = np.unravel_index(int(np.nanargmin(score)), score.shape)

    i, j = int(best_index[0]), int(best_index[1])

    minimum = (float(x_values[i]), float(y_values[j]))

    if i == 0 or j == 0 or i == len(x_values) - 1 or j == len(y_values) - 1:

        return {

            'angle_deg': float('nan'),

            'minimum': minimum,

            'curvature_min': float('nan'),

            'curvature_max': float('nan'),

        }

    dx = float(x_values[1] - x_values[0])

    dy = float(y_values[1] - y_values[0])

    dxx = (score[i + 1, j] - 2.0 * score[i, j] + score[i - 1, j]) / (dx * dx)

    dyy = (score[i, j + 1] - 2.0 * score[i, j] + score[i, j - 1]) / (dy * dy)

    dxy = (score[i + 1, j + 1] - score[i + 1, j - 1] - score[i - 1, j + 1] + score[i - 1, j - 1]) / (4.0 * dx * dy)

    hessian = np.array([[dxx, dxy], [dxy, dyy]], dtype=float)

    eigenvalues, eigenvectors = np.linalg.eigh(hessian)

    trough_vector = eigenvectors[:, int(np.argmin(eigenvalues))]

    angle_deg = math.degrees(math.atan2(float(trough_vector[1]), float(trough_vector[0])))

    return {

        'angle_deg': float(angle_deg),

        'minimum': minimum,

        'curvature_min': float(np.min(eigenvalues)),

        'curvature_max': float(np.max(eigenvalues)),

    }





def compare_fields(result_a: dict[str, object], result_b: dict[str, object], metric_key: str) -> dict[str, object]:

    signed = metric_key in SIGNED_METRICS

    field_a = np.asarray(result_a['fields'][metric_key], dtype=float)

    field_b = np.asarray(result_b['fields'][metric_key], dtype=float)

    diff = normalize_field(field_a, signed=signed) - normalize_field(field_b, signed=signed)

    orientation_a = trough_orientation(field_a, np.asarray(result_a['x_values']), np.asarray(result_a['y_values']), signed=signed)

    orientation_b = trough_orientation(field_b, np.asarray(result_b['x_values']), np.asarray(result_b['y_values']), signed=signed)

    delta = ((float(orientation_b['angle_deg']) - float(orientation_a['angle_deg']) + 90.0) % 180.0) - 90.0

    return {

        'difference_field': diff,

        'orientation_a': orientation_a,

        'orientation_b': orientation_b,

        'angle_delta_deg': float(delta),

        'alignment_rms': float(np.sqrt(np.nanmean(diff * diff))),

    }





def plot_field_panel(ax, result: dict[str, object], metric_key: str, transform_kwargs: dict[str, float | bool], norm, cmap_name: str, signed: bool, show_contours: bool, show_grid: bool) -> None:

    values = np.asarray(result['fields'][metric_key], dtype=float)

    xs = np.asarray(result['x_values'], dtype=float)

    ys = np.asarray(result['y_values'], dtype=float)

    x0, x1 = grid_extent(xs)

    y0, y1 = grid_extent(ys)

    image = ax.imshow(values.T, origin='lower', extent=[x0, x1, y0, y1], aspect='auto', cmap=cmap_name, norm=norm)

    image.set_transform(build_affine(**transform_kwargs) + ax.transData)

    corners = np.array([[x0, y0], [x0, y1], [x1, y0], [x1, y1]], dtype=float)

    corners = apply_affine(corners, **transform_kwargs)

    pad_x = 0.08 * max(1.0e-6, float(np.ptp(corners[:, 0])))

    pad_y = 0.08 * max(1.0e-6, float(np.ptp(corners[:, 1])))

    ax.set_xlim(float(np.min(corners[:, 0]) - pad_x), float(np.max(corners[:, 0]) + pad_x))

    ax.set_ylim(float(np.min(corners[:, 1]) - pad_y), float(np.max(corners[:, 1]) + pad_y))

    if show_contours:

        xx, yy = np.meshgrid(xs, ys, indexing='ij')

        contour = ax.contour(xx, yy, values, levels=8, colors='white', linewidths=0.6, alpha=0.45)

        for collection in contour.collections:

            collection.set_transform(build_affine(**transform_kwargs) + ax.transData)

    target_point = apply_affine(np.array([result['target_xy']], dtype=float), **transform_kwargs)[0]

    best_index = np.unravel_index(int(np.nanargmin(np.abs(values) if signed else values)), values.shape)

    best_point = apply_affine(np.array([[xs[best_index[0]], ys[best_index[1]]]], dtype=float), **transform_kwargs)[0]

    overlay = apply_affine(np.asarray(result['overlay_vertices'], dtype=float), **transform_kwargs)

    ax.plot(*overlay.T, color='black', linewidth=1.1, alpha=0.9)

    ax.plot([overlay[-1, 0], overlay[0, 0]], [overlay[-1, 1], overlay[0, 1]], color='black', linewidth=1.1, alpha=0.9)

    ax.scatter([target_point[0]], [target_point[1]], marker='x', s=80, color='black', linewidths=1.8)

    ax.scatter([best_point[0]], [best_point[1]], marker='*', s=180, color='white', edgecolors='black', linewidths=0.8, zorder=4)

    ax.grid(show_grid, alpha=0.15)

    ax.set_title(f"{result['descriptor']['label']}\n{METRIC_LABELS[metric_key]}", fontsize=11)

    ax.set_xlabel(result['space'])

    ax.set_ylabel('score axis 2')

    return image


In [8]:
# 6. Add Slider Controls for Rotation, Translation, and Scale

case_a_preset = widgets.Dropdown(options=[('1-1-1 control', 'iso111'), ('4-5-6 small target', 'acute456')], value='iso111', description='Case A')

case_b_preset = widgets.Dropdown(options=[('1-1-1 control', 'iso111'), ('4-5-6 small target', 'acute456')], value='acute456', description='Case B')

case_a_label = widgets.Text(value=BASE_CASES['iso111'].label, description='A label')

case_b_label = widgets.Text(value=BASE_CASES['acute456'].label, description='B label')

case_a_r1 = widgets.FloatSlider(min=0.6, max=8.0, step=0.01, value=BASE_CASES['iso111'].truth_r1, description='A r1', continuous_update=False, readout_format='.3f')

case_a_r2 = widgets.FloatSlider(min=0.6, max=8.5, step=0.01, value=BASE_CASES['iso111'].truth_r2, description='A r2', continuous_update=False, readout_format='.3f')

case_b_r1 = widgets.FloatSlider(min=0.6, max=8.0, step=0.01, value=BASE_CASES['acute456'].truth_r1, description='B r1', continuous_update=False, readout_format='.3f')

case_b_r2 = widgets.FloatSlider(min=0.6, max=8.5, step=0.01, value=BASE_CASES['acute456'].truth_r2, description='B r2', continuous_update=False, readout_format='.3f')



space_selector = widgets.Dropdown(options=[('r1-r2 space', 'rgrid'), ('tau space', 'tau')], value='rgrid', description='space')

metric_selector = widgets.Dropdown(

    options=[

        ('total score', 'total_score'),

        ('corr score', 'corr_score'),

        ('ratio score', 'ratio_score'),

        ('log residual v/4', 'v4_log_residual'),

        ('log residual u/4', 'u4_log_residual'),

        ('log residual w/4', 'w4_log_residual'),

        ('log residual (v/4)/(u/4)', 'v4_over_u4_log_residual'),

        ('log residual (w/4)/(u/4)', 'w4_over_u4_log_residual'),

    ],

    value='total_score',

    description='metric',

)

grid_size_slider = widgets.IntSlider(min=15, max=41, step=2, value=25, description='grid', continuous_update=False)

anchor_slider = widgets.FloatSlider(min=4.0, max=16.0, step=1.0, value=8.0, description='anchor', continuous_update=False)

mp_dps_slider = widgets.IntSlider(min=30, max=60, step=5, value=40, description='mp dps', continuous_update=False)



rotation_slider = widgets.FloatSlider(min=-180.0, max=180.0, step=1.0, value=0.0, description='rotate', continuous_update=False)

translate_x_slider = widgets.FloatSlider(min=-2.0, max=2.0, step=0.05, value=0.0, description='shift x', continuous_update=False)

translate_y_slider = widgets.FloatSlider(min=-2.0, max=2.0, step=0.05, value=0.0, description='shift y', continuous_update=False)

scale_slider = widgets.FloatSlider(min=0.6, max=1.8, step=0.05, value=1.0, description='scale', continuous_update=False)

reflect_x_toggle = widgets.Checkbox(value=False, description='reflect x')

reflect_y_toggle = widgets.Checkbox(value=False, description='reflect y')

show_contours_toggle = widgets.Checkbox(value=True, description='show contours')

show_grid_toggle = widgets.Checkbox(value=False, description='show grid')

positive_cmap_selector = widgets.Dropdown(options=['viridis_r', 'magma', 'cividis', 'plasma'], value='viridis_r', description='cmap')





def load_case_widgets(case_key: str, label_widget, r1_widget, r2_widget) -> None:

    case = BASE_CASES[case_key]

    label_widget.value = case.label

    r1_widget.value = case.truth_r1

    r2_widget.value = case.truth_r2





case_a_preset.observe(lambda change: load_case_widgets(change['new'], case_a_label, case_a_r1, case_a_r2), names='value')

case_b_preset.observe(lambda change: load_case_widgets(change['new'], case_b_label, case_b_r1, case_b_r2), names='value')



controls_left = widgets.VBox([case_a_preset, case_a_label, case_a_r1, case_a_r2, case_b_preset, case_b_label, case_b_r1, case_b_r2])

controls_middle = widgets.VBox([space_selector, metric_selector, grid_size_slider, anchor_slider, mp_dps_slider])

controls_right = widgets.VBox([

    rotation_slider,

    translate_x_slider,

    translate_y_slider,

    scale_slider,

    reflect_x_toggle,

    reflect_y_toggle,

    show_contours_toggle,

    show_grid_toggle,

    positive_cmap_selector,

])

control_box = widgets.HBox([controls_left, controls_middle, controls_right])


In [ ]:
# 7. Wire Real-Time Updates with Efficient Recalculation

LAST_RENDER: dict[str, object] = {}





def build_comparison_figure(

    case_a_key: str,

    case_a_label_value: str,

    case_a_r1_value: float,

    case_a_r2_value: float,

    case_b_key: str,

    case_b_label_value: str,

    case_b_r1_value: float,

    case_b_r2_value: float,

    space: str,

    metric_key: str,

    grid_size: int,

    anchor_denominator: float,

    mp_dps: int,

    rotation_deg: float,

    translate_x: float,

    translate_y: float,

    scale_factor: float,

    reflect_x: bool,

    reflect_y: bool,

    show_contours: bool,

    show_grid: bool,

    positive_cmap: str,

):

    case_a = make_case_descriptor(case_a_key, case_a_label_value, case_a_r1_value, case_a_r2_value)

    case_b = make_case_descriptor(case_b_key, case_b_label_value, case_b_r1_value, case_b_r2_value)

    builder = build_rgrid if space == 'rgrid' else build_tau_grid

    result_a = builder(case_a, grid_size=grid_size, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

    result_b = builder(case_b, grid_size=grid_size, anchor_denominator=anchor_denominator, mp_dps=mp_dps)

    signed = metric_key in SIGNED_METRICS

    field_a = np.asarray(result_a['fields'][metric_key], dtype=float)

    field_b = np.asarray(result_b['fields'][metric_key], dtype=float)

    if signed:

        limit = max(float(np.nanmax(np.abs(field_a))), float(np.nanmax(np.abs(field_b))), 1.0e-12)

        norm = TwoSlopeNorm(vcenter=0.0, vmin=-limit, vmax=limit)

        cmap_name = 'RdBu_r'

    else:

        limit = max(float(np.nanmax(field_a)), float(np.nanmax(field_b)), 1.0e-12)

        norm = Normalize(vmin=0.0, vmax=limit)

        cmap_name = positive_cmap

    transform_kwargs = {

        'rotation_deg': rotation_deg,

        'translate_x': translate_x,

        'translate_y': translate_y,

        'scale_factor': scale_factor,

        'reflect_x': reflect_x,

        'reflect_y': reflect_y,

    }

    comparison = compare_fields(result_a, result_b, metric_key)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    image_a = plot_field_panel(axes[0, 0], result_a, metric_key, transform_kwargs, norm, cmap_name, signed, show_contours, show_grid)

    image_b = plot_field_panel(axes[0, 1], result_b, metric_key, transform_kwargs, norm, cmap_name, signed, show_contours, show_grid)

    diff = np.asarray(comparison['difference_field'], dtype=float)

    diff_limit = max(float(np.nanmax(np.abs(diff))), 1.0e-12)

    diff_norm = TwoSlopeNorm(vcenter=0.0, vmin=-diff_limit, vmax=diff_limit)

    axes[1, 0].imshow(diff.T, origin='lower', extent=[-1.0, 1.0, -1.0, 1.0], aspect='auto', cmap='coolwarm', norm=diff_norm)

    axes[1, 0].set_title('normalized difference (A - B)', fontsize=11)

    axes[1, 0].set_xlabel('local x')

    axes[1, 0].set_ylabel('local y')

    axes[1, 1].axis('off')

    metrics_text = '\n'.join(

        [

            'Orientation metrics',

            '',

            f"A trough angle: {comparison['orientation_a']['angle_deg']:.2f} deg",

            f"B trough angle: {comparison['orientation_b']['angle_deg']:.2f} deg",

            f"angle delta: {comparison['angle_delta_deg']:+.2f} deg",

            f"alignment RMS: {comparison['alignment_rms']:.4f}",

            '',

            f"A minimum: {comparison['orientation_a']['minimum']}",

            f"B minimum: {comparison['orientation_b']['minimum']}",

            '',

            f"A tau_eff: ({case_a['tau'].real:.6f}, {case_a['tau'].imag:.6f})",

            f"B tau_eff: ({case_b['tau'].real:.6f}, {case_b['tau'].imag:.6f})",

            '',

            f"A reference lattice: {case_a['reference_geometry']}",

            f"B reference lattice: {case_b['reference_geometry']}",

        ]

    )

    axes[1, 1].text(0.0, 1.0, metrics_text, va='top', ha='left', family='monospace', fontsize=10)

    fig.colorbar(image_a, ax=[axes[0, 0], axes[0, 1]], fraction=0.03, pad=0.02, label=METRIC_LABELS[metric_key])

    fig.colorbar(axes[1, 0].images[0], ax=axes[1, 0], fraction=0.046, pad=0.04, label='normalized difference')

    fig.suptitle('Interactive modular heatmap comparison', fontsize=15)

    payload = {

        'case_a': case_a,

        'case_b': case_b,

        'space': space,

        'metric_key': metric_key,

        'comparison': comparison,

        'transform': transform_kwargs,

        'grid_size': int(grid_size),

        'anchor_denominator': float(anchor_denominator),

        'mp_dps': int(mp_dps),

    }

    return fig, payload





def render_interactive_view(**kwargs):

    fig, payload = build_comparison_figure(**kwargs)

    LAST_RENDER.clear()

    LAST_RENDER['params'] = dict(kwargs)

    LAST_RENDER['payload'] = payload

    display(fig)

    plt.close(fig)





interactive_output = widgets.interactive_output(

    render_interactive_view,

    {

        'case_a_key': case_a_preset,

        'case_a_label_value': case_a_label,

        'case_a_r1_value': case_a_r1,

        'case_a_r2_value': case_a_r2,

        'case_b_key': case_b_preset,

        'case_b_label_value': case_b_label,

        'case_b_r1_value': case_b_r1,

        'case_b_r2_value': case_b_r2,

        'space': space_selector,

        'metric_key': metric_selector,

        'grid_size': grid_size_slider,

        'anchor_denominator': anchor_slider,

        'mp_dps': mp_dps_slider,

        'rotation_deg': rotation_slider,

        'translate_x': translate_x_slider,

        'translate_y': translate_y_slider,

        'scale_factor': scale_slider,

        'reflect_x': reflect_x_toggle,

        'reflect_y': reflect_y_toggle,

        'show_contours': show_contours_toggle,

        'show_grid': show_grid_toggle,

        'positive_cmap': positive_cmap_selector,

    },

)



display(Markdown('## Interactive comparison\nAdjust the target couplings and view transform, then watch the tau-space or coupling-space heatmaps update.'))

display(control_box)

display(interactive_output)


In [ ]:
# 8. Display Orientation Metrics and Difference Heatmaps

def summarize_last_render() -> dict[str, object]:

    if not LAST_RENDER:

        raise RuntimeError('Render the interactive view first.')

    payload = LAST_RENDER['payload']

    summary = {

        'space': payload['space'],

        'metric_key': payload['metric_key'],

        'case_a': payload['case_a']['label'],

        'case_b': payload['case_b']['label'],

        'angle_delta_deg': payload['comparison']['angle_delta_deg'],

        'alignment_rms': payload['comparison']['alignment_rms'],

        'minimum_a': payload['comparison']['orientation_a']['minimum'],

        'minimum_b': payload['comparison']['orientation_b']['minimum'],

    }

    print(json.dumps(summary, indent=2, default=str))

    return summary





display(Markdown('Use `summarize_last_render()` after moving the sliders to dump the current angle delta, minima, and normalized difference summary.'))


In [ ]:
# 9. Capture the Current View and Export Notebook State

snapshot_tag = widgets.Text(value='modular_snapshot', description='tag')

save_button = widgets.Button(description='Save snapshot', button_style='info')

save_output = widgets.Output()





def save_current_view(_=None):

    with save_output:

        save_output.clear_output()

        if not LAST_RENDER:

            print('Render a comparison first.')

            return

        tag = snapshot_tag.value.strip().replace(' ', '_') or datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

        png_path = NOTEBOOK_OUTPUT_DIR / f'{tag}.png'

        json_path = NOTEBOOK_OUTPUT_DIR / f'{tag}.json'

        fig, payload = build_comparison_figure(**LAST_RENDER['params'])

        fig.savefig(png_path, dpi=180)

        plt.close(fig)

        export_payload = {

            'saved_at_utc': datetime.now(timezone.utc).isoformat(),

            'params': LAST_RENDER['params'],

            'payload': {

                'space': payload['space'],

                'metric_key': payload['metric_key'],

                'case_a': payload['case_a'],

                'case_b': payload['case_b'],

                'comparison': payload['comparison'],

                'transform': payload['transform'],

                'grid_size': payload['grid_size'],

                'anchor_denominator': payload['anchor_denominator'],

                'mp_dps': payload['mp_dps'],

            },

            'artifacts': {

                'png': str(png_path),

            },

        }

        json_path.write_text(json.dumps(export_payload, indent=2, default=str), encoding='utf-8')

        LAST_RENDER['last_saved'] = {'png': str(png_path), 'json': str(json_path)}

        print('saved:', png_path)

        print('state:', json_path)





save_button.on_click(save_current_view)

display(widgets.HBox([snapshot_tag, save_button]))

display(save_output)


In [ ]:
# 10. Write Continuation Context to a Journal File

journal_note = widgets.Textarea(

    value='Observed orientation difference and next step notes go here.',

    description='notes',

    layout=widgets.Layout(width='100%', height='120px'),

)

journal_button = widgets.Button(description='Append continuation entry', button_style='success')

journal_output = widgets.Output()





def append_notebook_journal(_=None):

    with journal_output:

        journal_output.clear_output()

        if not LAST_RENDER:

            print('Render a comparison first.')

            return

        payload = LAST_RENDER['payload']

        saved = LAST_RENDER.get('last_saved', {})

        timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

        lines = [

            '',

            f'## {timestamp}: Interactive modular notebook snapshot',

            '',

            f"- space: {payload['space']}",

            f"- metric: {payload['metric_key']}",

            f"- case A: {payload['case_a']['label']} with (r1, r2)=({payload['case_a']['r1']:.6f}, {payload['case_a']['r2']:.6f})",

            f"- case B: {payload['case_b']['label']} with (r1, r2)=({payload['case_b']['r1']:.6f}, {payload['case_b']['r2']:.6f})",

            f"- case A tau_eff: ({payload['case_a']['tau'].real:.12f}, {payload['case_a']['tau'].imag:.12f})",

            f"- case B tau_eff: ({payload['case_b']['tau'].real:.12f}, {payload['case_b']['tau'].imag:.12f})",

            f"- angle delta (B - A): {payload['comparison']['angle_delta_deg']:+.4f} deg",

            f"- alignment RMS: {payload['comparison']['alignment_rms']:.6f}",

            f"- transform: {json.dumps(payload['transform'])}",

            f"- reference lattices: A={payload['case_a']['reference_geometry']}, B={payload['case_b']['reference_geometry']}",

            f"- saved artifacts: {saved if saved else 'none yet'}",

            '',

            '### Notes',

            '',

            journal_note.value.strip(),

            '',

        ]

        with JOURNAL_ARTIFACT.open('a', encoding='utf-8') as handle:

            handle.write('\n'.join(lines))

        print('appended continuation entry to', JOURNAL_ARTIFACT)





journal_button.on_click(append_notebook_journal)

display(Markdown(f'Continuation entries from this notebook will be appended to `{JOURNAL_ARTIFACT}`.'))

display(widgets.VBox([journal_note, journal_button, journal_output]))
